In [ ]:
from datasets import load_dataset
import whisper
import random
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from transformers import pipeline
import requests
import json
import time


from transformers.models.whisper.english_normalizer import BasicTextNormalizer
import evaluate

normalizer = BasicTextNormalizer()
wer_metric = evaluate.load("wer")

def normalize_text(text: str) -> str:
    """Normalize text using Whisper's basic text normalizer."""
    return normalizer(text.strip())

def compute_wer(reference: str, prediction: str) -> float:
    """Compute WER between two strings after normalization."""
    norm_ref = normalize_text(reference)
    norm_pred = normalize_text(prediction)
    return wer_metric.compute(references=[norm_ref], predictions=[norm_pred])

# Load the dataset
dataset = load_dataset("google/fleurs", "lb_lu")
samples = dataset["test"]

prepared_samples = []

import tempfile
import soundfile as sf
import torch
import torchaudio
from tqdm import tqdm

for sample in tqdm(samples):
    audio_array = sample["audio"]["array"]
    sample_rate = sample["audio"]["sampling_rate"]
    reference = sample["transcription"].strip()

    # Resample if necessary
    if sample_rate != 16000:
        audio_array = torchaudio.functional.resample(
            torch.tensor(audio_array), orig_freq=sample_rate, new_freq=16000
        ).numpy()

    # Save to temp file
    tmp_file = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    sf.write(tmp_file.name, audio_array, 16000)

    prepared_samples.append({
        "path": tmp_file.name,
        "reference": reference
    })

/home/tunwellens/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 100/100 [00:02<00:00, 37.52it/s]


In [ ]:
# Load the Whisper model
model = whisper.load_model("medium")

refs_preds = []
wers = []

for sample in tqdm(prepared_samples):
    # Transcribe the audio
    result = model.transcribe(sample["path"], language="lb", task="transcribe")
    prediction = result["text"].strip()

    # Compute WER
    error = compute_wer(sample["reference"], prediction)
    wers.append(error)
    refs_preds.append((sample["reference"], prediction, error))

# calculate average WER and print results
average_wer = sum(wers) / len(wers)
print(f"Average WER: {average_wer:.2%}")

# show a few random samples

for ref, pred, err in random.sample(refs_preds, min(5, len(refs_preds))):
    print(f"Reference: {ref}")
    print(f"Predicted: {pred}")
    print(f"WER: {err:.2%}\n")

100%|██████████| 100/100 [08:39<00:00,  5.19s/it]

Average WER: 94.77%
Reference: op e puer strecken hunn déi gréisst entreprisen hir eege fligeren awer fir aner strecken a méi kleng entreprisë gouf et e problem
Predicted: Ab bu pua strakken nun de gris antarprisenn je eje fliegeren, aber fia anna strakken, am e klang antarprisenn, goviddi problem.
WER: 95.65%

Reference: veronstalt de site net andeem dir d'strukture mat graffiti markéiert oder zerkraazt
Predicted: F'n steeidt ja die Sit accustomed hoo strak doctors 김ach go
WER: 100.00%

Reference: d'danielle lantagne eng un-expertin fir dës krankheet huet duergeluecht datt d'friddenstruppe warscheinlech fir den ausbroch verantwortlech sinn
Predicted: Daniel Anton ain wánx xpærti seit h 핑es krank et hu油 d sage lušň dat tuv'rydn ma schall incvi Ок da t h knit xpäess� tatt danes tu í truprit wa şal admit m is konitt해도 vàage skfud na enjoyable
WER: 200.00%

Reference: den del potro hat de fréie virdeel am zweete saz awer och deen huet en tie-break nom erreeche vu 6:6 gebraucht
Predicted: 

### As we can see from this, Whisper Medium forced to transcribe luxembourgish performs very poorly with a WER (Word Error Ration) of 94.27%.

In [3]:
refs_preds = []
wers = []

for sample in tqdm(prepared_samples):
    # Transcribe the audio
    result = model.transcribe(sample["path"], task="transcribe")
    prediction = result["text"].strip()

    # Compute WER
    error = compute_wer(sample["reference"], prediction)
    wers.append(error)
    refs_preds.append((sample["reference"], prediction, error))

# calculate average WER and print results
average_wer = sum(wers) / len(wers)
print(f"Average WER: {average_wer:.2%}")

# show a few random samples

for ref, pred, err in random.sample(refs_preds, min(5, len(refs_preds))):
    print(f"Reference: {ref}")
    print(f"Predicted: {pred}")
    print(f"WER: {err:.2%}\n")

100%|██████████| 100/100 [09:42<00:00,  5.82s/it]

Average WER: 94.67%
Reference: jiddereen deen an héije breetegraden oder iwwer e biergpass fiert sollt d'méiglechkeet vu schnéi äis oder gefréierenden temperaturen bedenken
Predicted: Sie verringern auch Hechschuhe, Breitegraden oder überall Bier, Spass, Fjords und Milch geht für Schnee, Eis oder gefrierende Temperaturen bedenken.
WER: 80.00%

Reference: se hänkt domat zesummen ass awer normalerweis net mat alpinschi-änlechen touren oder biergsteige verbonnen woubäi dat lescht op géiem terrain stattfënnt a vill méi steif schi a schong erfuerdert
Predicted: Sie hängt immer zusammen, also aber normalerweise nicht mit Alpeni-Ski, ähnlichen Touren oder Biersteige verbunden, wobei dort leicht obgehend Terrain stattfindet, auf Filme steif Ski auch schon erfordert.
WER: 90.00%

Reference: hie gouf vum vizepremierminister vu singapur wong kan seng begréisst an huet mam premierminister vu singapur lee hsien loong iwwer handels an terrorugeleeënheeten diskutéiert
Predicted: יגו פעם פרמייר מיניסט

### Without specifying the language to the Whisper Medium Model, letting it automatically detect the language, resulted in even worse predictions where it couldnt even detect the language in most cases, and outputed full sentences in unrelated languages. A complete failure in transcribing luxembourgish.

In [ ]:
# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model and processor
model_name = "Lemswasabi/wav2vec2-base-luxembourgish-4h"
processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2ForCTC.from_pretrained(model_name).to(device).eval()

# Store results
refs_preds = []
wers = []

# Evaluate
for sample in tqdm(prepared_samples):
    # Load resampled audio from disk
    speech_array, _ = sf.read(sample["path"])  # already 16kHz from preprocessing

    # Tokenize
    inputs = processor(speech_array, sampling_rate=16000, return_tensors="pt", padding=True)
    input_values = inputs.input_values.to(device)
    attention_mask = inputs.attention_mask.to(device) if "attention_mask" in inputs else None

    # Inference
    with torch.no_grad():
        logits = model(input_values, attention_mask=attention_mask).logits \
            if attention_mask is not None else model(input_values).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.decode(predicted_ids[0], skip_special_tokens=True).strip()

    # Compute WER
    reference = sample["reference"]
    error = compute_wer(reference, transcription)

    wers.append(error)
    refs_preds.append((reference, transcription, error))

# calculate average WER and print results
average_wer = sum(wers) / len(wers)
print(f"Average WER: {average_wer:.2%}")

# show a few random samples

for ref, pred, err in random.sample(refs_preds, min(5, len(refs_preds))):
    print(f"Reference: {ref}")
    print(f"Predicted: {pred}")
    print(f"WER: {err:.2%}\n")


Invalid model-index. Not loading eval results into CardData.
100%|██████████| 100/100 [00:05<00:00, 17.48it/s]

Average WER: 59.97%
Reference: dës theorie betruechten wat et mat bestëmmte leit op sech huet déi déi saache wëllen déi se wëllen a wéi eng saachen an hirer ëmgéigend dozou féieren datt se bestëmmt saache wëllen oder net
Predicted: dës theorie betruechten wan et et mat bestëmte leit op sech huet déi déi sache wëlen déi i se wëlen a wéi eng sachen an hirer am géigender ze féieren dat se bestëmt serwële er hnet
WER: 50.00%

Reference: al kulturen a vëlker hu se fir en einfachen zougang zu mëllech hoer fleesch an haut gehalen
Predicted: al kulturen a fëlker hu se fir en einfachen zougang zu mëlech hoer flech an haus gehalen
WER: 23.53%

Reference: mat kundalini-yoga gëtt d'kundalini-energie erliichtungsenergie duerch yogastellungen otemübungen mantraen a visualiséierungen erwächt
Predicted: mat koundalinijongerget dcondalinin energe erleschtungsenerge dhuerch chorgerstaung otmong eweuen mantraen a liseluserung kawescht
WER: 80.00%

Reference: et kann och vu virdeel sinn eng wild card ze k

In [ ]:
# List of pgilles Whisper models fine-tuned on Luxembourgish
whisper_models = [
    "Tun-Wellens/pgilles-whisper-tiny-lb",
    "Tun-Wellens/pgilles-whisper-medium-lb",
    "Tun-Wellens/pgilles-whisper-large-lb",
]

for whisper_model_name in whisper_models:
    print(f"\n Evaluating {whisper_model_name}")

    asr = pipeline(
        "automatic-speech-recognition",
        model=whisper_model_name,
        device=0 if torch.cuda.is_available() else -1
    )

    refs_preds = []
    wers = []
    skipped = 0

    for sample in tqdm(prepared_samples, desc=f"Whisper Eval: {whisper_model_name}"):
        try:
            # Load already-preprocessed 16kHz audio from file
            speech_array, _ = sf.read(sample["path"])

            # Run transcription
            result = asr(
                {"array": speech_array, "sampling_rate": 16000}
            )
            transcription = result["text"].strip()

            # Compute WER
            reference = sample["reference"]
            error = compute_wer(reference, transcription)

            wers.append(error)
            refs_preds.append((reference, transcription, error))

        except ValueError as e:
            if "more than 3000 mel input features" in str(e):
                skipped += 1
                continue
            else:
                raise e  # re-raise unexpected errors

    # Average WER
    if wers:
        average_wer_pgilles = sum(wers) / len(wers)
        print(f"\n Average WER ({whisper_model_name}) over {len(wers)} valid samples: {average_wer_pgilles:.2%}")
    else:
        print(f"\n No valid samples for {whisper_model_name}")

    if skipped:
        print(f" Skipped {skipped} sample(s) due to long-form constraint (>30s)")

    # Show sample predictions
    if refs_preds:
        print(f"\n Sample predictions ({whisper_model_name}):\n")
        for ref, pred, err in random.sample(refs_preds, min(5, len(refs_preds))):
            print(f"Reference: {ref}")
            print(f"Predicted: {pred}")
            print(f"WER: {err:.2%}\n")



 Evaluating pgilles/whisper-tiny-lb


Device set to use cuda:0
Whisper Eval: pgilles/whisper-tiny-lb:   0%|          | 0/100 [00:00<?, ?it/s]/home/tunwellens/.local/lib/python3.12/site-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.
Whisper Eval: pgilles/whisper-tiny-lb: 100%|██████████| 100/100 [00:45<00:00,  2.22it/s]



 Average WER (pgilles/whisper-tiny-lb) over 99 valid samples: 69.32%
 Skipped 1 sample(s) due to long-form constraint (>30s)

 Sample predictions (pgilles/whisper-tiny-lb):

Reference: op e puer strecken hunn déi gréisst entreprisen hir eege fligeren awer fir aner strecken a méi kleng entreprisë gouf et e problem
Predicted: Op e puer Strecken hunn dee gréisst Entreprisen hier eege Fligeren. Awer fir aner Staken a méi kleng Entreprisen gouf et e Problem.
WER: 17.39%

Reference: veronstalt de site net andeem dir d'strukture mat graffiti markéiert oder zerkraazt
Predicted: Verhonnstellt de Site vun der nodeem déi administratoire mat Grafitee mer gëllt dat zakrat.
WER: 84.62%

Reference: d'danielle lantagne eng un-expertin fir dës krankheet huet duergeluecht datt d'friddenstruppe warscheinlech fir den ausbroch verantwortlech sinn
Predicted: D' Daniel Lambon eng UN Expert fir dës Krankheet huet duerchgeluecht, datt d' Friddensdru beherrändlech fir den Ausbauverwaltung fortgeluecht sinn.
WE

Device set to use cuda:0
Whisper Eval: pgilles/whisper-large-v2-lb_cased_01: 100%|██████████| 100/100 [07:02<00:00,  4.23s/it]



 Average WER (pgilles/whisper-large-v2-lb_cased_01) over 99 valid samples: 61.85%
 Skipped 1 sample(s) due to long-form constraint (>30s)

 Sample predictions (pgilles/whisper-large-v2-lb_cased_01):

Reference: op e puer strecken hunn déi gréisst entreprisen hir eege fligeren awer fir aner strecken a méi kleng entreprisë gouf et e problem
Predicted: Op e puer Strecken hunn déi gréissten Entreprisen hir eege Fligeren awer fir aner Strecken a méi kleng Entreprisen gouf et e Problem.
WER: 8.70%

Reference: veronstalt de site net andeem dir d'strukture mat graffiti markéiert oder zerkraazt
Predicted: D'Verunstallte Site hunn et an deem déi Strukturen mat Graffitti, Markéierplaz a Kass.
WER: 92.31%

Reference: d'danielle lantagne eng un-expertin fir dës krankheet huet duergeluecht datt d'friddenstruppe warscheinlech fir den ausbroch verantwortlech sinn
Predicted: Danielle Lamport eng UN Experte fir dëst Krankheet huet duerchgeluecht datt d'Friddensgruppe waarschinlech fir den Ausfro verant

Device set to use cuda:0
Whisper Eval: pgilles/whisper-medium-v2-lb_cased_01: 100%|██████████| 100/100 [1:37:13<00:00, 58.33s/it]


 Average WER (pgilles/whisper-medium-v2-lb_cased_01) over 100 valid samples: 44.00%

 Sample predictions (pgilles/whisper-medium-v2-lb_cased_01):

Reference: op e puer strecken hunn déi gréisst entreprisen hir eege fligeren awer fir aner strecken a méi kleng entreprisë gouf et e problem
Predicted: Op e puer Strecken hunn déi gréis Entreprisen hir eege Fligeren, awer fir aner Strecken a méi kleng Entreprisen gouf et e Problem.
WER: 8.70%

Reference: veronstalt de site net andeem dir d'strukture mat graffiti markéiert oder zerkraazt
Predicted: Verunstelt se sitten et an deem dee Strukture mat graffitegem och gëtt a zekratzt.
WER: 92.31%

Reference: d'danielle lantagne eng un-expertin fir dës krankheet huet duergeluecht datt d'friddenstruppe warscheinlech fir den ausbroch verantwortlech sinn
Predicted: Даниëlle Lanton, ан ON-Expertin, huet duerchgeluecht, datt d'Friddensdruppe ass assent d'Ausbroch verantwortlech sinn.
WER: 65.00%

Reference: den del potro hat de fréie virdeel am zweete 

In [ ]:
# LuxASR endpoint
LUXASR_API = "https://luxasr.uni.lu/v2/asr?diarization=Disabled&outfmt=text"

refs_preds = []
wers = []

for sample in tqdm(prepared_samples):
    with open(sample["path"], "rb") as audio_file:
        files = {"audio_file": ("audio.wav", audio_file, "audio/wav")}
        response = requests.post(LUXASR_API, files=files, timeout=30)
        predicted = json.loads(response.text.strip())

    # Compute WER
    error = compute_wer(sample["reference"], predicted)
    wers.append(error)
    refs_preds.append((sample["reference"], predicted, error))

    time.sleep(1)  

# Average WER
average_wer = sum(wers) / len(wers)
print(f"\n Average WER (LuxASR) over {len(wers)} samples: {average_wer:.2%}")

# Show sample predictions
print("\n Sample predictions (LuxASR):\n")
for ref, pred, err in random.sample(refs_preds, min(5, len(refs_preds))):
    print(f"Reference: {ref}")
    print(f"Predicted: {pred}")
    print(f"WER: {err:.2%}\n")

100%|██████████| 100/100 [03:00<00:00,  1.80s/it]


 Average WER (LuxASR) over 100 samples: 22.95%

 Sample predictions (LuxASR):

Reference: op e puer strecken hunn déi gréisst entreprisen hir eege fligeren awer fir aner strecken a méi kleng entreprisë gouf et e problem
Predicted: Op e puer Strecken hunn déi gréisst Entreprisen hir eege Fligeren, awer fir aner Strecken a méi kleng Entreprisen gouf et e Problem.
WER: 4.35%

Reference: veronstalt de site net andeem dir d'strukture mat graffiti markéiert oder zerkraazt
Predicted: Vir ons stellt de Site net an deem dir d'Strukture mat Graffiti markéiert oder zerkraazt.
WER: 38.46%

Reference: d'danielle lantagne eng un-expertin fir dës krankheet huet duergeluecht datt d'friddenstruppe warscheinlech fir den ausbroch verantwortlech sinn
Predicted: Danielle Anton, eng ON-Expertin fir dës Krankheet, huet duerchgeluecht, datt Zefriddenstruppe warscheinlech fir den Ausbroch verantwortlech sinn.
WER: 30.00%

Reference: den del potro hat de fréie virdeel am zweete saz awer och deen huet en tie-br